# Prepare Analysis-Ready Land ACC Inputs

This notebook is the required preprocessing stage for `1b_refactor_land_leadtime_acc_skill_map.ipynb`. It transforms raw E3SM land hindcasts and an external observational product into compatible seasonal, 1° latitude–longitude NetCDF inputs. CPC Soil Moisture V2 is a model-calculated reference rather than a direct measurement; see the [NOAA PSL product page](https://psl.noaa.gov/data/gridded/data.cpcsoil.html) and [CPC model explanation](https://www.cpc.ncep.noaa.gov/soilmst/descrip.htm).

The output contract records the field, units, grid, seasonal convention, source product, anomaly status, evaluation/climatology protocol, ensemble size, and—for `H2OSOI`—the common physical depth interval. It also verifies that every represented model grid cell has all requested members. Run this notebook once per field/reference configuration, then run `1b` for metrics and plots.

In [9]:
%load_ext autoreload
%autoreload 2

import glob
import os
import re
import uuid
from pathlib import Path

import numpy as np
import xarray as xr

from esp_lab import data_access_e3sm, land_skill
from esp_lab.paths import S2D_DIAG_ROOT
from esp_lab.utils import regrid_utils as regrid


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Processing configuration

Set `field`, the raw reference configuration, and the verified observational depth support when processing `H2OSOI`. The CPC Soil Moisture V2 default is a one-layer, model-calculated water-height equivalent in a 1.6 m soil column—not a 0–10 cm volumetric measurement.

In [10]:
field = "H2OSOI"  # H2OSNO, TWS, or H2OSOI

SOIL_DEPTH_RANGE_M = (0.0, 1.6) if field == "H2OSOI" else None
FORCE_REWRITE = True
TARGET_DLAT = 1.0
TARGET_DLON = 1.0

RUN = {
    # Match 1a_refactor_leadtime_acc_skill_map.ipynb.
    "years": (1980, 2018),
    "climatology_years": (1981, 2010),
    "init_months": [5, 11],
    "nens": 10,
    "monthly_nlead": 24,
    "strict_member_completeness": True,
}

E3SM_CASES = {
    "E3SM-FOSIRL": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL",
        "cache_tag": "JRA55_FOSIRL",
    },
    "E3SM-Reanalysis": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce",
        "cache_tag": "Reanalysis",
    },
}

data_dir_candidates = [
    Path("/global/cfs/cdirs/e3sm/S2S2D/post_process"),
    Path("/global/cfs/cdirs/e3smdata/simulations/S2S2D/post_process"),
]
DATA_DIR = next((path for path in data_dir_candidates if path.exists()), data_dir_candidates[0])
PROCESSED_ROOT = Path(
    os.environ.get(
        "ESP_LAB_LAND_ACC_INPUT_ROOT",
        str(Path(S2D_DIAG_ROOT) / "land_acc_inputs"),
    )
)
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)


In [11]:
REFERENCE_CONFIGS = {
    "H2OSNO": {
        "path": os.environ.get("ESP_LAB_H2OSNO_REFERENCE", ""),
        "variable": "swe",
        "product": "CONFIGURE_ME",
        "scale": 1.0,
        "offset": 0.0,
        "output_units": "mm",
        "is_anomaly": False,
        "already_seasonal": False,
    },
    "TWS": {
        "path": os.environ.get("ESP_LAB_TWS_REFERENCE", ""),
        "variable": "tws",
        "product": "CONFIGURE_ME",
        "scale": 1.0,
        "offset": 0.0,
        "output_units": "mm",
        "is_anomaly": True,
        "already_seasonal": False,
    },
    "H2OSOI": {
        "path": os.environ.get(
            "ESP_LAB_H2OSOI_REFERENCE",
            "/global/cfs/cdirs/e3sm/zhan391/data/CPC_SOM/monthly/soilw_*.nc",
        ),
        "variable": "soilw",
        "product": "CPC_Soil_Moisture_V2",
        "documentation": "https://www.cpc.ncep.noaa.gov/soilmst/descrip.htm",
        "scale": 1.0,
        "offset": 0.0,
        "output_units": "mm",
        "is_anomaly": False,
        "already_seasonal": False,
        # Layered reference: set both values and ensure bounds are metres.
        "vertical_dim": None,
        "layer_bounds_variable": None,
        # Already depth-averaged reference: document its actual interval.
        "represented_depth_range_m": (0.0, 1.6),
    },
}
reference_cfg = REFERENCE_CONFIGS[field]
if reference_cfg["product"] == "CONFIGURE_ME":
    raise ValueError("Set the reference product name and verify all reference settings")
reference_cfg

{'path': '/global/cfs/cdirs/e3sm/zhan391/data/CPC_SOM/monthly/soilw_*.nc',
 'variable': 'soilw',
 'product': 'CPC_Soil_Moisture_V2',
 'documentation': 'https://www.cpc.ncep.noaa.gov/soilmst/descrip.htm',
 'scale': 1.0,
 'offset': 0.0,
 'output_units': 'mm',
 'is_anomaly': False,
 'already_seasonal': False,
 'vertical_dim': None,
 'layer_bounds_variable': None,
 'represented_depth_range_m': (0.0, 1.6)}

## Shared helpers and output contract

In [12]:
def safe_token(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")


def depth_tag():
    if field != "H2OSOI":
        return ""
    top, bottom = SOIL_DEPTH_RANGE_M
    return f"_depth{top:g}-{bottom:g}m_integrated_mm"


PRODUCT_TAG = safe_token(reference_cfg["product"])
GRID_TAG = f"{TARGET_DLAT:g}x{TARGET_DLON:g}deg_cell_centered"


def model_output_path(case_tag, init_month):
    directory = PROCESSED_ROOT / "model" / case_tag / field
    return directory / f"{case_tag}{init_month:02d}_{field}{depth_tag()}_seasonal_{GRID_TAG}.nc"


def reference_output_path():
    directory = PROCESSED_ROOT / "reference" / PRODUCT_TAG / field
    return directory / f"{PRODUCT_TAG}_{field}{depth_tag()}_seasonal_{GRID_TAG}.nc"


def safe_to_netcdf(ds, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp.{uuid.uuid4().hex}")
    try:
        ds.to_netcdf(temporary)
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()


def contract_attrs(source_kind, source_name, is_anomaly=False):
    attrs = {
        "processing_stage": "analysis_ready_land_acc_input_v1",
        "field": field,
        "source_kind": source_kind,
        "source_name": source_name,
        "horizontal_grid": GRID_TAG,
        "seasonal_convention": "centered_3month_DJF_MAM_JJA_SON",
        "reference_is_anomaly": str(bool(is_anomaly)).lower(),
        "evaluation_protocol": "1a_refactor_init1980-2018_clim1981-2010_per-lead_v1",
        "initialization_years": f"{RUN['years'][0]}-{RUN['years'][1]}",
        "climatology_years": f"{RUN['climatology_years'][0]}-{RUN['climatology_years'][1]}",
        "ensemble_size": RUN["nens"],
        "monthly_leads": RUN["monthly_nlead"],
    }
    if field == "H2OSOI":
        attrs["depth_top_m"], attrs["depth_bottom_m"] = SOIL_DEPTH_RANGE_M
        top, bottom = SOIL_DEPTH_RANGE_M
        attrs["vertical_aggregation"] = f"{top:g}-{bottom:g}m water-height equivalent"
    return attrs


## Load and vertically/temporally harmonize the raw reference

In [13]:
def standardize_reference_coordinates(da):
    rename = {}
    for source, target in [("latitude", "lat"), ("longitude", "lon")]:
        if source in da.dims or source in da.coords:
            rename[source] = target
    da = da.rename(rename)
    required = {"time", "lat", "lon"}
    available = set(da.dims) | set(da.coords)
    if not required.issubset(available):
        raise ValueError(f"Reference is missing coordinates: {sorted(required - available)}")
    da = da.assign_coords(lon=(da.lon % 360)).sortby("lon")
    return da.sortby("lat")


def load_raw_reference(cfg):
    path_text = str(Path(cfg["path"]).expanduser()) if cfg["path"] else ""
    has_glob = any(char in path_text for char in "*?[]")
    paths = sorted(glob.glob(path_text)) if has_glob else (
        [path_text] if path_text and Path(path_text).exists() else []
    )
    if not paths:
        raise FileNotFoundError(f"No raw reference files match {cfg['path']!r}")
    ds = xr.open_mfdataset(paths, combine="by_coords") if len(paths) > 1 else xr.open_dataset(paths[0])
    if cfg["variable"] not in ds:
        raise KeyError(f"{cfg['variable']!r} not found; variables={list(ds.data_vars)}")
    da = standardize_reference_coordinates(ds[cfg["variable"]]).chunk(
        {"time": 24, "lat": 90, "lon": 180}
    )
    da = (da * cfg["scale"] + cfg["offset"]).rename(field)
    da.attrs["units"] = cfg["output_units"]

    if field == "H2OSOI":
        vertical_dim = cfg.get("vertical_dim")
        if vertical_dim is not None:
            bounds_name = cfg.get("layer_bounds_variable")
            if not bounds_name or bounds_name not in ds:
                raise ValueError("Layered reference requires a valid layer_bounds_variable")
            da = land_skill.depth_weighted_soil_moisture(
                da, SOIL_DEPTH_RANGE_M, vertical_dim=vertical_dim,
                layer_bounds_m=ds[bounds_name],
            )
        else:
            represented = cfg.get("represented_depth_range_m")
            if represented is None:
                raise ValueError("Document represented_depth_range_m for the 2-D reference")
            if not np.allclose(represented, SOIL_DEPTH_RANGE_M, rtol=0.0, atol=1e-6):
                raise ValueError(
                    f"Reference depth {represented} differs from target {SOIL_DEPTH_RANGE_M}"
                )
            da.attrs["depth_top_m"], da.attrs["depth_bottom_m"] = SOIL_DEPTH_RANGE_M

    reference_attrs = dict(da.attrs)
    if not cfg["already_seasonal"]:
        da = da.rolling(time=3, center=True, min_periods=3).mean()
        da = da.where(da.time.dt.month.isin([1, 4, 7, 10]), drop=True)
        da = da.dropna("time", how="all")
        da.attrs.update(reference_attrs)
    return da


reference_native = load_raw_reference(reference_cfg)
reference_native

<xarray.DataArray 'H2OSOI' (time: 309, lat: 180, lon: 360)> Size: 80MB
dask.array<getitem, shape=(309, 180, 360), dtype=float32, chunksize=(22, 90, 180), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 2kB 1948-04-01 1948-07-01 ... 2025-04-01
  * lat      (lat) float64 1kB -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
  * lon      (lon) float64 3kB 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
Attributes:
    standard_name:   lwe_thickness_of_soil_moisture_content
    long_name:       Model-Calculated Monthly Mean Soil Moisture
    units:           mm
    dataset:         CPC Monthly Soil Moisture
    var_desc:        Soil Moisture
    level_desc:      Surface
    statistic:       Monthly Mean
    parent_stat:     Other
    cell_methods:    time: mean (monthly from values)
    actual_range:    [  0.     756.0375]
    depth_top_m:     0.0
    depth_bottom_m:  1.6

## Regrid and write the analysis-ready reference

In [14]:
destgrid = xr.Dataset(
    coords={
        "lat": np.arange(-90.0 + TARGET_DLAT / 2, 90.0, TARGET_DLAT),
        "lon": np.arange(TARGET_DLON / 2, 360.0, TARGET_DLON),
    }
)


def is_target_grid(da):
    return (
        np.array_equal(da.lat.values, destgrid.lat.values)
        and np.array_equal(da.lon.values, destgrid.lon.values)
    )


if is_target_grid(reference_native):
    reference_ready = reference_native.copy()
else:
    reference_regridder = regrid.make_regridder(
        reference_native.to_dataset(name=field), destgrid,
        method="conservative", periodic=True,
    )
    reference_ready = reference_regridder(reference_native).rename(field)
reference_ready.attrs.update(reference_native.attrs)
reference_ready.attrs.update(
    contract_attrs("reference", reference_cfg["product"], reference_cfg["is_anomaly"])
)
reference_ready.attrs["units"] = reference_cfg["output_units"]
reference_ready.attrs["documentation"] = reference_cfg.get("documentation", "")
reference_path = reference_output_path()
if FORCE_REWRITE or not reference_path.exists():
    safe_to_netcdf(reference_ready.to_dataset(name=field), reference_path)
print("Reference output:", reference_path)


Reference output: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/land_acc_inputs/reference/CPC_Soil_Moisture_V2/H2OSOI/CPC_Soil_Moisture_V2_H2OSOI_depth0-1.6m_integrated_mm_seasonal_1x1deg_cell_centered.nc


## Load, vertically aggregate, seasonally average, regrid, and write E3SM inputs

In [15]:
years = np.arange(RUN["years"][0], RUN["years"][1] + 1)
members = [f"EN{i:02d}" for i in range(RUN["nens"])]
monthly_chunks = {"Y": 3, "L": 24, "M": 2, "lat": 90, "lon": 180}
model_outputs = {}
model_regridder = None

for case_name, case_cfg in E3SM_CASES.items():
    model_outputs[case_name] = {}
    for init_month in RUN["init_months"]:
        output_path = model_output_path(case_cfg["cache_tag"], init_month)
        model_outputs[case_name][init_month] = output_path
        if output_path.exists() and not FORCE_REWRITE:
            print("Reuse:", output_path)
            continue

        monthly = land_skill.load_e3sm_land_monthly(
            data_dir=str(DATA_DIR), case_prefix=case_cfg["case_prefix"],
            members=members, init_tags=data_access_e3sm.build_init_tags(years, init_month),
            field=field, nlead=RUN["monthly_nlead"], chunks=monthly_chunks,
        )
        selection = (
            {
                "soil_depth_range_m": SOIL_DEPTH_RANGE_M,
                "soil_output": "water_equivalent_mm",
            }
            if field == "H2OSOI" else {}
        )
        seasonal = land_skill.seasonal_land_hindcast_dataset(monthly, field, **selection)
        land_skill.validate_hindcast_evaluation_setup(
            seasonal[field], seasonal.time, init_month=init_month,
            initialization_years=RUN["years"], expected_members=members,
            climatology_years=RUN["climatology_years"],
            require_complete_member_grid=RUN["strict_member_completeness"],
        )
        if is_target_grid(seasonal[field]):
            ready_field = seasonal[field].copy()
        else:
            if model_regridder is None:
                model_regridder = regrid.make_regridder(
                    seasonal, destgrid, method="conservative", periodic=True
                )
            ready_field = model_regridder(seasonal[field]).rename(field)
        ready_field.attrs.update(seasonal[field].attrs)
        ready_field.attrs.update(contract_attrs("model", case_name))
        ready_field.attrs.setdefault("units", land_skill.LAND_VARIABLES[field].units)
        ready = ready_field.to_dataset(name=field)
        ready["time"] = seasonal.time
        ready.attrs.update(contract_attrs("model", case_name))
        safe_to_netcdf(ready, output_path)
        print("Wrote:", output_path)


Wrote: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/land_acc_inputs/model/JRA55_FOSIRL/H2OSOI/JRA55_FOSIRL05_H2OSOI_depth0-1.6m_integrated_mm_seasonal_1x1deg_cell_centered.nc
Wrote: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/land_acc_inputs/model/JRA55_FOSIRL/H2OSOI/JRA55_FOSIRL11_H2OSOI_depth0-1.6m_integrated_mm_seasonal_1x1deg_cell_centered.nc
Wrote: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/land_acc_inputs/model/Reanalysis/H2OSOI/Reanalysis05_H2OSOI_depth0-1.6m_integrated_mm_seasonal_1x1deg_cell_centered.nc
Wrote: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/land_acc_inputs/model/Reanalysis/H2OSOI/Reanalysis11_H2OSOI_depth0-1.6m_integrated_mm_seasonal_1x1deg_cell_centered.nc


## Validate the staged handoff

This reopens every output, verifies the required contract, and compares each model field with the staged reference. `1b` repeats these inexpensive checks before computing metrics.

In [16]:
with xr.open_dataset(reference_path) as ds:
    reference_check = ds[field].load()

for case_name, by_month in model_outputs.items():
    for init_month, path in by_month.items():
        with xr.open_dataset(path, chunks={"Y": -1, "L": 4, "M": 2, "lat": 45, "lon": 90}) as ds:
            required = {field, "time"}
            if not required.issubset(ds.variables):
                raise ValueError(f"{path} is missing {sorted(required - set(ds.variables))}")
            if ds[field].attrs.get("processing_stage") != "analysis_ready_land_acc_input_v1":
                raise ValueError(f"{path} has the wrong processing contract")
            model_field, model_time, dropped = land_skill.retain_valid_seasonal_leads(
                ds[field], ds.time
            )
            model_time = model_time.load()
            expected_years = land_skill.validate_hindcast_evaluation_setup(
                model_field, model_time, init_month=init_month,
                initialization_years=RUN["years"], expected_members=members,
                climatology_years=RUN["climatology_years"],
                require_complete_member_grid=RUN["strict_member_completeness"],
            )
            model_check = model_field.isel(Y=0, M=0).load()
        land_skill.validate_land_reference_compatibility(model_check, reference_check)
        land_skill.validate_reference_time_coverage(
            reference_check, expected_years, model_time, RUN["climatology_years"]
        )
        print("Validated:", case_name, init_month, path, "dropped leads:", dropped)

print("Preprocessing complete. Run 1b_refactor_land_leadtime_acc_skill_map.ipynb next.")

Validated: E3SM-FOSIRL 5 /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/land_acc_inputs/model/JRA55_FOSIRL/H2OSOI/JRA55_FOSIRL05_H2OSOI_depth0-1.6m_integrated_mm_seasonal_1x1deg_cell_centered.nc dropped leads: []
Validated: E3SM-FOSIRL 11 /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/land_acc_inputs/model/JRA55_FOSIRL/H2OSOI/JRA55_FOSIRL11_H2OSOI_depth0-1.6m_integrated_mm_seasonal_1x1deg_cell_centered.nc dropped leads: []


Validated: E3SM-Reanalysis 5 /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/land_acc_inputs/model/Reanalysis/H2OSOI/Reanalysis05_H2OSOI_depth0-1.6m_integrated_mm_seasonal_1x1deg_cell_centered.nc dropped leads: []
Validated: E3SM-Reanalysis 11 /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/land_acc_inputs/model/Reanalysis/H2OSOI/Reanalysis11_H2OSOI_depth0-1.6m_integrated_mm_seasonal_1x1deg_cell_centered.nc dropped leads: []
Preprocessing complete. Run 1b_refactor_land_leadtime_acc_skill_map.ipynb next.
